In [3]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader
 
from lib.dataset.dataloader import AMSR2Dataset, collate_crop_to_min, collate_pad_to_max
from lib.model.Baseline import EncDec

In [ ]:
## Config — match your training setup ###
CACHE_DIR   = '/dmidata/projects/asip-cms/ninna_msc/zarr_cache'
OUTPUT_DIR  = "/dmidata/users/nili/Master/Master-thesis---Super-resolution-sea-ice-concentration-using-generative-AI/outputs/training/baseline"
CKPT_PATH   = os.path.join(OUTPUT_DIR, 'best_baseline_model_2.pth')
# SAVE_PATH   = os.path.join(OUTPUT_DIR, 'prediction_with_amsr2.png')
NUM_WORKERS = 16

 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
 
### Load checkpoint — all model and training params come from here ###
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
 
# Required for model reconstruction
in_channels = ckpt['in_channels']
features    = ckpt['features']
 
# Optional training params — shown in plot title if present
batch_size  = ckpt.get('batch_size',    '?')
lr          = ckpt.get('learning_rate', '?')
wd          = ckpt.get('weight_decay',  '?')
grad_clip   = ckpt.get('grad_clip_norm','?')
collate     = ckpt.get('collate',       '?')
if collate == 'crop_to_min':
    COLLATE_FN = collate_crop_to_min
else:
    COLLATE_FN = collate_pad_to_max
 
print(f"Checkpoint  : epoch={ckpt['epoch']}  in_channels={in_channels}  features={features}")
print(f"Val metrics : loss={ckpt['val_loss']:.4f}  rmse={ckpt['val_rmse']:.2f}%  mae={ckpt['val_mae']:.2f}%")
print(f"Training    : lr={lr}  batch={batch_size}  wd={wd}  grad_clip={grad_clip}  collate={collate}")
 
### Build model from checkpoint params ###
model = EncDec(in_channels=in_channels, features=features).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
 
### Load dataset — batch_size from checkpoint if available ###
bs = batch_size if isinstance(batch_size, int) else 8
val_dataset = AMSR2Dataset(CACHE_DIR, split='val')
val_loader  = DataLoader(
    val_dataset, batch_size=bs, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    collate_fn=COLLATE_FN,
)
 
### Run inference ###
amsr2, sic, mask = next(iter(val_loader))
amsr2, sic, mask = amsr2.to(device), sic.to(device), mask.to(device)
target_size = (sic.shape[-2], sic.shape[-1])
 
with torch.no_grad():
    pred = model(amsr2, target_size=target_size)
 

In [ ]:
SAMPLE_IDX  = 0   # which sample in the batch to plot

### Extract sample ###
idx      = SAMPLE_IDX
amsr2_np = amsr2[idx].cpu().numpy()   # (14, H, W)
sic_np   = sic[idx, 0].cpu().numpy()  # (H, W)
mask_np  = mask[idx, 0].cpu().numpy() # (H, W) bool
pred_np  = pred[idx, 0].cpu().numpy() # (H, W)
 
sic_np  = np.where(mask_np, np.nan, sic_np)
pred_np = np.where(mask_np, np.nan, pred_np)
diff_np = pred_np - sic_np

plot

In [ ]:
channel_names = val_dataset.channel_names  # list of 14 names from zarr attrs
 
cmap_bt   = plt.cm.Greys.copy(); cmap_bt.set_bad('beige')
cmap_sic  = plt.cm.Blues_r.copy();  cmap_sic.set_bad('lightgray')
cmap_diff = plt.cm.bwr_r.copy();   cmap_diff.set_bad('lightgray')
 
fig = plt.figure(figsize=(24, 18))
gs  = gridspec.GridSpec(4, 5, figure=fig, hspace=0.35, wspace=0.3)
 
# ── AMSR2 channels: rows 0-2, 5 columns (14 channels + 1 empty) ──────────────
for i in range(in_channels):
    row, col = divmod(i, 5)
    ax = fig.add_subplot(gs[row, col])
    im = ax.imshow(amsr2_np[i], cmap=cmap_bt, interpolation='nearest')
    ax.set_title(channel_names[i] if i < len(channel_names) else f'ch {i}', fontsize=8)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label='BT (K)')
 
fig.add_subplot(gs[2, 4]).axis('off')  # 15th cell — empty
 
# ── SIC target / prediction / error: row 3 ───────────────────────────────────
vmin, vmax = 0, 100
 
ax0 = fig.add_subplot(gs[3, 0])
im0 = ax0.imshow(sic_np, cmap=cmap_sic, vmin=vmin, vmax=vmax, interpolation='nearest')
ax0.set_title('Target SIC (%)', fontsize=10)
ax0.axis('off')
plt.colorbar(im0, ax=ax0, fraction=0.046, pad=0.02, label='SIC (%)')
 
ax1 = fig.add_subplot(gs[3, 1])
im1 = ax1.imshow(pred_np, cmap=cmap_sic, vmin=vmin, vmax=vmax, interpolation='nearest')
ax1.set_title('Predicted SIC (%)', fontsize=10)
ax1.axis('off')
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02, label='SIC (%)')
 
abs_max = np.nanmax(np.abs(diff_np))
ax2 = fig.add_subplot(gs[3, 2])
im2 = ax2.imshow(diff_np, cmap=cmap_diff, vmin=-abs_max, vmax=abs_max, interpolation='nearest')
ax2.set_title('Prediction Error (%)', fontsize=10)
ax2.axis('off')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.02, label='Δ SIC (%)')
 
fig.add_subplot(gs[3, 3]).axis('off')
fig.add_subplot(gs[3, 4]).axis('off')
 
fig.suptitle(
    f'AMSR2 inputs (rows 1–3)  |  SIC target, prediction, error (row 4)  — sample {idx}\n'
    f'Epoch {ckpt["epoch"]}  |  '
    f'val_loss={ckpt["val_loss"]:.4f}  val_rmse={ckpt["val_rmse"]:.2f}%  val_mae={ckpt["val_mae"]:.2f}%  |  '
    f'lr={lr}  batch={batch_size}  collate={collate}',
    fontsize=10,
)
 
# plt.savefig(SAVE_PATH, dpi=150, bbox_inches='tight')
# plt.close()
# print(f"Saved → {SAVE_PATH}")